# The Agent Memory Lottery
**Companion notebook for the 2026-10-08 post.** A memory plugin stores 1,000 snippets and
retrieves by vector similarity on every prompt. This notebook reproduces the two core
failure modes on a toy codebase and measures them:

1. **Stale recall**: similarity search cannot tell a superseded fragment from the current one.
2. **Accuracy decay**: facts change weekly; snippets are never pruned.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- Toy "memory plugin" contents: snippets scraped from session transcripts ---
# The auth fact changed in week 6: API-key header -> OAuth2 + PKCE. Both versions live on.
fragments = [
    "authentication uses x api key header sent with every request authentication token key header backend api",
    "auth api key header x api key token authentication backend key auth api key header",
    "oauth2 pkce redirect flow refresh tokens code challenge verifier sign in no keys",
    "oauth pkce code verifier redirect uri sign in tokens refresh flow",
    "db postgres primary connection string pool size 20 production database",
    "db sqlite in memory for tests postgres only production database tests",
    "deploy region us east 1 primary eks cluster us east",
    "deploy region eu west 1 primary cluster migrated from us east deploys",
]
ids = ["auth-v1 (STALE)", "auth-v1 (STALE)", "auth-v2 (CURRENT)", "auth-v2 (CURRENT)",
       "db prod", "db tests", "deploy v1 (STALE)", "deploy v2 (CURRENT)"]

# --- Doc workspace: ONE current page per topic (the 'documentation' answer) ---
docs = {
    "auth.md": "Authentication: OAuth2 with PKCE. No API keys. Redirect flow, refresh tokens.",
    "db.md": "DB: Postgres in production, SQLite in-memory for tests.",
    "deploy.md": "Deploy: primary cluster is eu-west-1 (migrated from us-east-1).",
}
query = "how does authentication work"
print("fragments:", len(fragments), "| docs:", len(docs))
print("query:", repr(query))


In [ ]:
# --- Retrieval by similarity (the memory-plugin retrieval step) ---
vec = TfidfVectorizer()
X = vec.fit_transform(fragments + [query])
sim = cosine_similarity(X[-1], X[:-1]).ravel()
rank = np.argsort(-sim)
print("Top-3 retrieved fragments for the auth query:")
for r in rank[:3]:
    print(f"  sim={sim[r]:.3f}  <- {ids[r]}")

# The stale fragment wins: it shares more surface wording (api, key, token, header)
# with the query than the current fragment does. Distance measures wording, not truth.
top_is_stale = "STALE" in ids[rank[0]]
print("\nTop hit is STALE:", top_is_stale)
assert top_is_stale, "expected the stale fragment to outrank the current one"


In [ ]:
# --- 12-week simulation: facts churn, fragments are never pruned ---
rng = np.random.RandomState(8)
per_week = 16   # queries answered each week
n_weeks = 12
churn = 0.20    # 20% of not-yet-changed facts get replaced each week (absorbing)

def trial(memory="rag"):
    current = rng.rand(50) < 0.5      # current truth
    stored = current.copy()           # what memory holds
    changed_once = np.zeros(50, bool)
    correct = 0
    for week in range(n_weeks):
        new = (rng.rand(50) < churn) & ~changed_once
        changed_once |= new
        current[new] = ~current[new]  # fact replaced; memory copy stays stale forever
        if memory == "doc":
            stored = current.copy()   # docs are updated when code changes
        qs = rng.choice(50, per_week)
        correct += (stored[qs] == current[qs]).sum()
    return correct / (n_weeks * per_week)

rag_acc = np.mean([trial("rag") for _ in range(20)])
doc_acc = np.mean([trial("doc") for _ in range(20)])
print(f"Correct-recall rate over 12 weeks: RAG memory = {rag_acc:.2%}, doc workspace = {doc_acc:.2%}")
assert doc_acc == 1.0, "doc workspace is always current"
assert rag_acc < 0.50, f"RAG memory should decay badly, got {rag_acc:.2%}"


## Takeaways

1. **Similarity is not a truth filter.** The stale auth fragment outscored the current one
   (0.87 vs 0.71 in the toy run) purely on word overlap — the exact failure mode the
   liao.gg essay flags as structural, not tunable.
2. **Staleness compounds.** With 20% weekly churn and no pruning, the share of still-true
   fragments is 0.8^w — about 13% by week 9, 11% by week 10. The simulation's 12-week
   average correct-recall rate is 30.6% for RAG memory vs 100% for the doc workspace.
3. **The doc-workspace loop is `prompt → consult → build → update`**, not
   `prompt → build → forget`. One current page per topic, read whole, reviewed like code —
   which is also what Meta's hourly Muse dossiers are doing at consumer scale, just with
   people instead of code.

**Exercise for the reader:** rerun cell 3 with `max_features` or a real sentence-embedding
model and check whether the stale fragment still wins — it usually does, which is the point.